In [1]:
# 1. Install plotly just in case the environment doesn't have it
!pip install plotly

import pandas as pd
import numpy as np
import plotly.express as px

# 2. Load the dataset (Colab can see it now because you uploaded it)
df = pd.read_csv('/content/dataset.csv')

# Define the lists based on the dataset columns
assets = [
    'Compound_V_Vials', 'Heisenberg_Crystals', 'Infinity_Stones',
    'Pieces_Of_Eight', 'Solaris_Stardust', 'Valyrian_Steel'
]
maestros = ['Jack', 'Queen', 'Ace', 'King', 'Black_Joker', 'Red_Joker']

# 3. Calculate Forward Returns for each asset
for asset in assets:
    close_col = f"{asset}_close"
    df[f"{asset}_fwd_ret"] = df[close_col].pct_change().shift(-1)

# 4. Initialize a DataFrame to store the results
hit_rate_matrix = pd.DataFrame(index=maestros, columns=assets, dtype=float)

# 5. Calculate the Hit Rate
for maestro in maestros:
    for asset in assets:
        ret_col = f"{asset}_fwd_ret"
        active_signals = df[df[maestro] != 0].dropna(subset=[ret_col])
        correct_predictions = (np.sign(active_signals[maestro]) == np.sign(active_signals[ret_col])).sum()
        total_signals = len(active_signals)

        if total_signals > 0:
            hit_rate = correct_predictions / total_signals
        else:
            hit_rate = 0.0

        hit_rate_matrix.loc[maestro, asset] = hit_rate

# 6. Plot the heatmap using Plotly
fig = px.imshow(
    hit_rate_matrix,
    text_auto=".2%",
    color_continuous_scale="Blues",
    title="Maestro Signal Hit Rate vs. Asset Forward Returns",
    labels=dict(x="Assets", y="Maestros", color="Hit Rate")
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()